# Simple Mean-Reverting Model with Open-to-Open Reversion, 3-Day Rebalancing and Top/Bottom-50 Selection

In [1]:
import numpy as np

In [2]:
import pandas as pd

In [3]:
import requests

In [4]:
from bs4 import BeautifulSoup

In [5]:
import yfinance as yf

## Universe: current S&P 600 SmallCap constituents, open prices from yfinance (2020-2025)

In [6]:
startDate='2020-01-01'

In [7]:
endDate='2025-12-31'

In [8]:
def get_sp600_tickers():
    url = 'https://en.wikipedia.org/wiki/List_of_S%26P_600_companies'
    resp = requests.get(url, headers={'User-Agent': 'Mozilla/5.0'}, timeout=30)
    soup = BeautifulSoup(resp.text, 'html.parser')
    table = soup.find('table', {'id': 'constituents'})
    rows = table.find_all('tr')[1:]
    return [row.find_all('td')[0].get_text(strip=True).replace('.', '-') for row in rows]

In [9]:
tickers=get_sp600_tickers()

In [10]:
def download_open_prices(tickers, start, end):
    data = yf.download(tickers, start=start, end=end, auto_adjust=False, threads=True, progress=False)
    return data['Open']

In [11]:
# download extra history before startDate so the first daily return in the analysis window isn't NaN
downloadStart=(pd.Timestamp(startDate)-pd.Timedelta(days=30)).strftime('%Y-%m-%d')
downloadEnd=(pd.Timestamp(endDate)+pd.Timedelta(days=1)).strftime('%Y-%m-%d')
df=download_open_prices(tickers, downloadStart, downloadEnd)

HTTP Error 404: {"quoteSummary":{"result":null,"error":{"code":"Not Found","description":"Quote not found for symbol: CWEN-A"}}}


$CWEN-A: possibly delisted; no timezone found


$LEG: Data doesn't exist for startDate = 1575262800, endDate = 1767243600


$MBGL: Data doesn't exist for startDate = 1575262800, endDate = 1767243600


$VGNT: Data doesn't exist for startDate = 1575262800, endDate = 1767243600


...

$HLX: Data doesn't exist for startDate = 1575262800, endDate = 1767243600


$MFP: Data doesn't exist for startDate = 1575262800, endDate = 1767243600



7 Failed downloads:


['CWEN-A']: possibly delisted; no timezone found


['LEG', 'MBGL', 'VGNT', 'ADIG', 'HLX', 'MFP']: Data doesn't exist for startDate = 1575262800, endDate = 1767243600


In [12]:
df.sort_index(inplace=True)

In [13]:
dailyret=df.pct_change()

In [14]:
marketDailyret=dailyret.mean(axis=1)

## Signal: cross-sectional z-score, rebalanced every 3rd trading day, top/bottom 50 performers held until |z| <= 0.5

In [15]:
devRet=dailyret.sub(marketDailyret, axis=0)

In [16]:
crossSectionalStd=devRet.std(axis=1)

In [17]:
zscore=devRet.div(crossSectionalStd, axis=0)

In [18]:
rebalanceDay=pd.Series(np.arange(len(dailyret.index)) % 3 == 0, index=dailyret.index) # rebalancing interval: every 3rd trading day

In [19]:
rankUnderperformers=zscore.rank(axis=1, ascending=True) # most negative deviation = strongest underperformer -> long

In [20]:
rankOutperformers=zscore.rank(axis=1, ascending=False) # most positive deviation = strongest outperformer -> short

In [21]:
entryLong=pd.DataFrame(False, index=dailyret.index, columns=dailyret.columns)
entryLong[rankUnderperformers<=50]=True
entryLong.loc[~rebalanceDay]=False # only enter new positions on rebalancing days

In [22]:
entryShort=pd.DataFrame(False, index=dailyret.index, columns=dailyret.columns)
entryShort[rankOutperformers<=50]=True
entryShort.loc[~rebalanceDay]=False

In [23]:
exitSignal=zscore.abs()<=0.5
exitSignal.loc[~rebalanceDay]=False # exit is also only evaluated on rebalancing days

In [24]:
positions_Long=pd.DataFrame(np.nan, index=dailyret.index, columns=dailyret.columns)

In [25]:
positions_Short=pd.DataFrame(np.nan, index=dailyret.index, columns=dailyret.columns)

In [26]:
positions_Long[entryLong]=1

In [27]:
positions_Short[entryShort]=-1

In [28]:
positions_Long[exitSignal]=0 # exit overrides entry

In [29]:
positions_Short[exitSignal]=0

In [30]:
positions_Long=positions_Long.ffill().fillna(0) # carry position forward until the next entry/exit signal

In [31]:
positions_Short=positions_Short.ffill().fillna(0)

In [32]:
positions=positions_Long+positions_Short

In [33]:
wtsum=positions.abs().sum(axis=1)

In [34]:
wtsum[wtsum==0]=1

In [35]:
weights=positions.div(wtsum, axis=0) # dollar-neutral normalization

In [36]:
dailypnl=np.nansum(np.array(pd.DataFrame(weights).shift())*np.array(dailyret), axis=1)

In [37]:
dailypnl=dailypnl[np.logical_and(df.index >= startDate, df.index <= endDate)]

In [38]:
sharpeRatio=np.sqrt(252)*np.mean(dailypnl)/np.std(dailypnl)

In [39]:
sharpeRatio

np.float64(1.0236723228346292)

# With transaction costs

In [40]:
onewaytcost=0.0005

In [41]:
weights=weights[np.logical_and(df.index >= startDate, df.index <= endDate)]

In [42]:
dailypnlminustcost=dailypnl - (np.nansum(abs(weights-np.array(pd.DataFrame(weights).shift())), axis=1)*onewaytcost)

In [43]:
sharpeRatioMinusTcost=np.sqrt(252)*np.mean(dailypnlminustcost)/np.std(dailypnlminustcost)

In [44]:
sharpeRatioMinusTcost

np.float64(0.508556545937279)